# Operações Vetoriais

Uma tarefa comum é querer considerar todos os valores dentro de uma série e fazer algum tipo de operação. Isso pode ser tentar encontrar um determinado número, resumir os dados ou transformá-los de alguma forma.

In [1]:
# Biblioteca
import pandas as pd

In [2]:
# Uma abordagem programática típica para isso seria repetir todos os itens da série e invocar a operação na qual estamos interessados. 
# Por exemplo, poderíamos criar uma série de números inteiros reprexentando as notas dos alunos e tentar obter a média.

notas = pd.Series([9, 8, 7, 6])
total = 0

total = sum(nota for nota in notas) / len(notas)

total



7.5

In [3]:
# Isso pode funcionar, mas podemos aproveitar mais do hardware. O Pandas suporta métodos de computação vetorial. Para isso, precisamos da biblioteca NumPy.

import numpy as np

# Agora chamaremos np.sum() e passaremos um iterável, isto é, nesse caso, nossa série do Pandas. 
total = np.sum(notas) / len(notas)
total

np.float64(7.5)

Os dois métodos mostrados produzem o mesmo resultado, mas a diferença é que com o método np.sum() estaremos utilizando operações vetoriais (uso de registradores vetoriais (SIMD) e é implementado em C), mas isso não significa que é garatido o uso de SIMD, mas o importante é que diminui o overhead do Python. Além disso, como são utilizados arrays do numpy, há possibilidade de termos o uso de memória contígua. <br>
Enquanto o sum() do próprio Python percorre elemento por elemento somando de forma iterativa

In [17]:
# Vamos criar uma grande série de números aleatórios para provar se np.sum() é mais rápido que sum()
numeros = pd.Series(np.random.rand(1000000))

# Vamos olhar os cinco primeiros itens desta série para garantir que eles existem
print(numeros.head())
print(len(numeros))

0    0.634393
1    0.517234
2    0.380207
3    0.604367
4    0.656448
dtype: float64
1000000


Agora que temos uma grande série. <br>
O interpretador do Python tem magic functions do Jupyter/IPython que começam com '%%'. Então vamos usar uma função mágica celular chamada 'timeit', em que essa função executará nosso código algumas vezes para determinar, em média, quanto tempo leva para executar a função. <br>

OBS: Podemos dar ao timeit o número de loops que gostaríamos de executar, mas por padrão são 1000 loops. <br>
OBS2: Para utilizar a função cellular magic, ela precisa ser a primeira linha da célula. <br>



As magic functions são comandos especiais do Jupyter/IPython que começam com % (linha) ou %% (célula). Elas não fazem parte do Python padrão, mas são recursos do ambiente interativo para facilitar tarefas como medir tempo, executar comandos do sistema, instalar pacotes e rodar código em outras linguagens.

- %  → atua em uma única linha (ex: %time, %ls, %pwd)
- %% → atua em toda a célula (ex: %%time, %%bash)

Exemplo: <br>
%time sum(range(1000000)) <br>
%%time <br>
sum(range(1000000)) <br>

Exemplo: <br>
%%timeit -n NUM_EXECUCOES -r NUM_REPETICOES_POR_EXECUCAO <br>

In [19]:
%%timeit -n 100

total = sum(numero for numero in numeros)
total / len(numeros)

94.1 ms ± 887 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [25]:
%%timeit -n 100

total = np.sum(numeros)
total / len(numeros)

1.14 ms ± 81.4 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Esse resultado vem do uso de computação paralela.

In [26]:
# Um recurso relacionado no Pandas e no NumPy é chamado de broadcasting. Com broadcasting, podemos aplicar uma operação a cada valor da serie, alterando a série de forma muito rápida (espalhando para todos os itens da série).
# Por exemplo, se quiséssemos aumentar cada variável aleatória em 2, poderíamos fazer isso rapidamente usando o operador += no objeto da série.

# Broadcasting permite operações entre:
# Vetor + Escalar
# Vetor + Vetor de mesmo tamanho
# Matriz + Vetor
# Matriz + Escalar
# OBS: Desde que as dimensões sejam compatíveis.

# Olhando as 5 primeiras linhas da série
numeros.head()

0    0.634393
1    0.517234
2    0.380207
3    0.604367
4    0.656448
dtype: float64

In [27]:
# Agora vamos aumentar tudo na série em 2 unidades

numeros += 2
numeros.head()

0    2.634393
1    2.517234
2    2.380207
3    2.604367
4    2.656448
dtype: float64

Aqui também tem o uso de SIMD e de memória contígua se as condições permitirem.

O que são condições que não permitem o uso de vetorização?
- A SIMD funciona melhor com tipos numéricos simples, como int e float, mas se for object, então não tem como utilizar SIMD. 
- Se os dados não estiverem em memória contígua, fica difícil carregá-los para o registrador vetorial. 
- SIMD funciona para operações simples, como +-/*
- E obviamente, a CPU precisa ter suporte para operações vetoriais.

In [37]:
# SE FOSSEMOS FAZER DE FORMA ITERATIVA, SERIA:
# Pandas suporta a iteração da série, assim como em um dicionário, o que nos permite descompactar valores facilmente.

# Vamos utilizar a função iteritems(), o que retorna um rótulo e um valor. 
for label, valor in numeros.items():
    # O item que é retornado, vamos chamar de set_value()
    numeros.loc[label] = valor + 2

numeros.head()

0    4.634393
1    4.517234
2    4.380207
3    4.604367
4    4.656448
dtype: float64

##### ATENÇÃO: Se você estiver iterando a qualquer momento no Pandas ou NumPy, se pergunte se está fazendo as coisas da melhor maneira possível

##### OBS: Podemos criar nossas próprias funções vetorizadas